In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care_Epi_contract_id

MPB-------------

In [ ]:
-- Populate care_epi_contr_id using the unique contract ID from the new RDM Contracts table
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join MPB tenancy contract to the new RDM Contracts table using source system instance and source contract ID
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND rdmc.contr_src_id = CONCAT('MPB ', CAST(ten.id AS varchar(100)))

In [ ]:
SELECT
    CONCAT('MPB', CAST(u.id AS varchar(100))) AS care_epi_src_id,
    u.id AS user_id,
    ten.id AS tenancy_source_id,
    CAST(ten.id AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_drj_users u
LEFT JOIN
    silver_drj_tenancies ten
    ON ten.id = u.tenancy_id
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(ten.id AS varchar(100))
WHERE
    u.profile_type = 'user'
LIMIT 100;

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(ten.id AS varchar(100))